In [1]:
using CSV
using Dates
using StringEncodings
using DataFrames

In [2]:
ENV["COLUMNS"] = 1000;

In [3]:
data_date = "2021_02_18";

In [4]:
hospitals = ["BMC", "HCGH", "JHH", "SH", "SMH"];

In [5]:
census_total_data = DataFrame(CSV.File(open(read, "../rawdata/realdata_$(data_date)/CensusWorksheet.csv", enc"UTF-16")));
active_total_data = filter(row -> !ismissing(row.CalcInstitutionName) && row.CalcInstitutionName != "JOHNS HOPKINS ALL CHILDREN'S HOSPITAL", census_total_data)
active_total_data = select(active_total_data,
    :CalcInstitutionName => (xs -> [string(split.(x, " ")[1]) for x in xs]) => :hospital,
    "Day of adate" => (x -> Date.(x, dateformat"U d, Y")) => :date,
    :CensusCount => :active_total,
);

In [6]:
census_icu_data = DataFrame(CSV.File(open(read, "../rawdata/realdata_2021_02_18/ICUCensus.csv", enc"UTF-16")));
active_icu_data = filter(row -> !ismissing(row.CalcInstitutionName) && row.CalcInstitutionName != "JOHNS HOPKINS ALL CHILDREN'S HOSPITAL", census_icu_data)
active_icu_data = select(active_icu_data,
    :CalcInstitutionName => (xs -> [string(split.(x, " ")[1]) for x in xs]) => :hospital,
    "Day of adate" => (x -> Date.(x, dateformat"U d, Y")) => :date,
    :CensusICUCount => :active_icu,
);

In [7]:
census_flagged_data = DataFrame(CSV.File(open(read, "../rawdata/realdata_$(data_date)/CensusWorksheetCOVIDActive.csv", enc"UTF-16")))
active_flagged_data = filter(row -> !ismissing(row.CalcInstitutionName) && row.CalcInstitutionName != "JOHNS HOPKINS ALL CHILDREN'S HOSPITAL", census_flagged_data)
active_flagged_data = select(active_flagged_data,
    :CalcInstitutionName => (xs -> [string(split.(x, " ")[1]) for x in xs]) => :hospital,
    "Day of adate" => (x -> Date.(x, dateformat"U d, Y")) => :date,
    :CensusCount => :active_total_flagged,
);

In [8]:
active_data = outerjoin(active_total_data, active_flagged_data, active_icu_data, on=[:hospital, :date])
for col in names(active_data)
    active_data[!,col] = coalesce.(active_data[!,col], 0)
end
active_data.active_acute = active_data.active_total - active_data.active_icu;

In [9]:
function load_admissions(hname)
    adm = DataFrame(CSV.File(open(read, "../rawdata/realdata_$(data_date)/$(hname)CumAdmits.csv", enc"UTF-16")))
    adm = stack(adm, r"adate*")
    
    z = maximum([x for x in names(adm) if contains(x, "Column")])
    adm[:,z] = [ismissing(x) ? "date" : x for x in adm[:,z]]
    
    adm = unstack(adm, :variable, z, :value)
    select!(adm,
        :date,
        "Admissions to JHMI (excludes transfers)" => :admissions,
    )
    adm.date = [Date(d, dateformat"m/d/yyyy") for d in adm.date]
    adm.admissions = [parse(Float64, x) for x in adm.admissions]
    adm.admissions = [Int(x) for x in adm.admissions]
    
    rename!(adm, :admissions => :admissions_total)
    
    return adm
end;

function load_admissions()
    data = []
    for hname in hospitals
        df = load_admissions(hname)
        insertcols!(df, 1, :hospital => fill(hname, nrow(df)))
        push!(data, df)
    end
    df = vcat(data...)
    return df
end;

In [10]:
admissions_data = load_admissions();

In [11]:
combined_data = outerjoin(active_data, admissions_data, on=[:hospital, :date])
for col in names(combined_data)
    combined_data[!,col] = coalesce.(combined_data[!,col], 0)
end
sort!(combined_data, [:hospital, :date]);

In [12]:
combined_data |> CSV.write("../data/jhhs_realdata_$(data_date).csv");